# Token Embedding Analysis

For each target layer we compute **running means** over the full ImageNet val set (no individual embeddings stored in RAM):

- `mean_cls[l]`   — mean CLS token embedding across all val images  
- `mean_patch[l]` — mean embedding per patch position (16×16 grid) across all val images  

Then we compute and plot:

1. **Spatial heatmap per layer** — `‖mean_cls[l] − mean_patch[l][i,j]‖₂` on the 16×16 patch grid  
2. **Scalar line chart across layers**
   - `‖mean(cls)‖₂` — norm of the average CLS embedding (consensus direction strength)  
   - `mean(‖cls‖₂)` — average of per-image CLS norms (typical magnitude)

All mean embeddings are saved to disk at the end.

In [ ]:
import torch
import timm
import timm.data
import numpy as np
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import os

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# ── Model selection — uncomment the model you want to use ──────────────────
MODEL_NAME = "vit_large_patch14_clip_224.openai_ft_in1k"   # CLIP ViT-L/14
# MODEL_NAME = "vit_large_patch16_224.augreg_in21k_ft_in1k"  # ViT-L/16 (AugReg)
# MODEL_NAME = "deit3_large_patch16_224.fb_in1k"             # DeiT3-L/16
# MODEL_NAME = "vit_large_patch14_dinov2.lvd142m"           # DINOv2-L/14    (518×518, 37×37 grid)
_MODEL_SHORT = MODEL_NAME.split('.')[0]  # used to namespace output paths

IMAGENET_ROOT = "/data/imagenet/extracted"   # path with val/ subdir
BATCH_SIZE    = 64
NUM_WORKERS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# Layers to analyse; None = all layers
TARGET_LAYERS = None   # e.g. [0, 6, 12, 18, 23] for a subset

# Where to save mean embeddings (read by 03_cls_lens_mean_adjusted_scoring.ipynb)
SAVE_PATH = f"../outputs/{_MODEL_SHORT}/precomputed_means.pt"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load model (frozen, eval mode)
model = timm.create_model(MODEL_NAME, pretrained=True)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

num_layers   = len(model.blocks)
d_model      = model.num_features   # works for all timm ViTs incl. DINOv2 (num_classes=0)
grid_size    = model.patch_embed.grid_size   # (H, W), e.g. (16, 16)
H, W         = grid_size

if TARGET_LAYERS is None:
    TARGET_LAYERS = list(range(num_layers))

print(f"Model      : {MODEL_NAME}")
print(f"Device     : {DEVICE}")
print(f"d_model    : {d_model}")
print(f"Patch grid : {H}×{W} = {H*W} patches")
print(f"Layers     : {TARGET_LAYERS}")

In [ ]:
# Register forward hooks — capture ALL tokens [B, seq_len, d_model] and move to CPU immediately
_hook_outputs = {}   # {layer_idx: Tensor[B, seq_len, d_model]}
_hooks = []

def _make_hook(layer_idx):
    def hook_fn(module, input, output):
        # Move to CPU right away to keep GPU memory free
        _hook_outputs[layer_idx] = output.detach().cpu()
    return hook_fn

for layer_idx in TARGET_LAYERS:
    handle = model.blocks[layer_idx].register_forward_hook(_make_hook(layer_idx))
    _hooks.append(handle)

print(f"Registered hooks on {len(_hooks)} layers.")

In [ ]:
# Build val dataloader
data_config   = timm.data.resolve_model_data_config(model)
val_transform = timm.data.create_transform(**data_config, is_training=False)

val_dataset = ImageFolder(f"{IMAGENET_ROOT}/val", transform=val_transform)
val_loader  = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Val images : {len(val_dataset):,}")
print(f"Val batches: {len(val_loader):,}")

In [ ]:
# ── Running-mean accumulators (CPU, float64 for numerical stability) ──────────
# sum_cls[l]       : [d_model]     — sum of CLS embeddings across images
# sum_patch[l]     : [H, W, d_model] — sum of each patch embedding across images
# sum_cls_norm[l]  : scalar        — sum of per-image ‖cls‖₂ values
# n_images         : int           — total images processed

sum_cls      = {l: torch.zeros(d_model,    dtype=torch.float64) for l in TARGET_LAYERS}
sum_patch    = {l: torch.zeros(H, W, d_model, dtype=torch.float64) for l in TARGET_LAYERS}
sum_cls_norm = {l: 0.0 for l in TARGET_LAYERS}
n_images     = 0

print("Accumulators initialised.")
patch_mem_mb = H * W * d_model * 8 * len(TARGET_LAYERS) / 1e6
print(f"Peak accumulator RAM: ~{patch_mem_mb:.1f} MB")

In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────────────
with torch.no_grad():
    for images, _ in tqdm(val_loader, desc="Accumulating means"):
        images = images.to(DEVICE)
        _hook_outputs.clear()
        _ = model(images)          # triggers all hooks

        B = images.size(0)
        n_images += B

        for l in TARGET_LAYERS:
            tokens = _hook_outputs[l]  # [B, 1+H*W, d_model], on CPU
            tokens = tokens.double()   # use float64 for accumulation

            cls_tokens   = tokens[:, 0, :]               # [B, d_model]
            patch_tokens = tokens[:, 1:, :].reshape(B, H, W, d_model)  # [B, H, W, d_model]

            sum_cls[l]       += cls_tokens.sum(dim=0)           # [d_model]
            sum_patch[l]     += patch_tokens.sum(dim=0)         # [H, W, d_model]
            sum_cls_norm[l]  += cls_tokens.norm(dim=-1).sum().item()

        _hook_outputs.clear()   # free memory immediately

# Remove hooks
for h in _hooks:
    h.remove()
_hooks.clear()

print(f"\nProcessed {n_images:,} images.")

In [ ]:
# ── Compute final means (back to float32) ─────────────────────────────────────
mean_cls      = {}   # {l: [d_model]}
mean_patch    = {}   # {l: [H, W, d_model]}
mean_cls_norm = {}   # {l: scalar}   — mean of per-image ‖cls‖₂
norm_mean_cls = {}   # {l: scalar}   — ‖mean(cls)‖₂

for l in TARGET_LAYERS:
    mean_cls[l]      = (sum_cls[l]   / n_images).float()
    mean_patch[l]    = (sum_patch[l] / n_images).float()
    mean_cls_norm[l] = sum_cls_norm[l] / n_images
    norm_mean_cls[l] = mean_cls[l].norm().item()

print("Means computed. Sample check (layer 0):")
l0 = TARGET_LAYERS[0]
print(f"  ‖mean_cls[{l0}]‖₂      = {norm_mean_cls[l0]:.4f}  (norm of mean CLS)")
print(f"  mean(‖cls‖₂)[{l0}]     = {mean_cls_norm[l0]:.4f}  (mean of norms)")

In [ ]:
# ── Save mean embeddings ───────────────────────────────────────────────────────
save_dict = {
    "mean_cls":      torch.stack([mean_cls[l]   for l in TARGET_LAYERS]),   # [L, d_model]
    "mean_patch":    torch.stack([mean_patch[l] for l in TARGET_LAYERS]),   # [L, H, W, d_model]
    "norm_mean_cls": [norm_mean_cls[l] for l in TARGET_LAYERS],             # list[float] len L
    "mean_cls_norm": [mean_cls_norm[l] for l in TARGET_LAYERS],             # list[float] len L
    "target_layers": TARGET_LAYERS,
    "model_name":    MODEL_NAME,
    "n_images":      n_images,
    "patch_grid":    (H, W),
}
torch.save(save_dict, SAVE_PATH)
print(f"Saved to {os.path.abspath(SAVE_PATH)}")
print(f"  mean_cls  : {save_dict['mean_cls'].shape}")
print(f"  mean_patch: {save_dict['mean_patch'].shape}")

In [ ]:
# ── Plot 1: Spatial heatmaps of ‖mean_cls[l] − mean_patch[l][i,j]‖₂ ──────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

n_layers = len(TARGET_LAYERS)
n_cols   = 6
n_rows   = (n_layers + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 2.8, n_rows * 2.8),
                         squeeze=False)

# Collect all heatmap values first to set a shared colour scale
all_heatmaps = []
for l in TARGET_LAYERS:
    diff = mean_cls[l].unsqueeze(0).unsqueeze(0) - mean_patch[l]   # [H, W, d_model]
    hmap = diff.norm(dim=-1).numpy()                                # [H, W]
    all_heatmaps.append(hmap)

vmin = min(h.min() for h in all_heatmaps)
vmax = max(h.max() for h in all_heatmaps)

for idx, (l, hmap) in enumerate(zip(TARGET_LAYERS, all_heatmaps)):
    row, col = divmod(idx, n_cols)
    ax = axes[row][col]
    im = ax.imshow(hmap, vmin=vmin, vmax=vmax, cmap="viridis", origin="upper")
    ax.set_title(f"Layer {l}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide unused subplots
for idx in range(n_layers, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row][col].set_visible(False)

# Shared colourbar
fig.subplots_adjust(right=0.88, hspace=0.35, wspace=0.1)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="‖mean_cls − mean_patch‖₂")

fig.suptitle(
    f"‖mean_cls[l] − mean_patch[l][i,j]‖₂  per layer\n{MODEL_NAME}",
    fontsize=12
)

plt.savefig("heatmap_cls_patch_diff.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to heatmap_cls_patch_diff.png")

In [ ]:
# ── Plot 2: ‖mean(cls)‖₂  vs  mean(‖cls‖₂)  across layers ───────────────────
layers_x      = TARGET_LAYERS
norm_of_mean  = [norm_mean_cls[l] for l in layers_x]   # ‖E[cls]‖₂
mean_of_norms = [mean_cls_norm[l] for l in layers_x]   # E[‖cls‖₂]

fig, ax = plt.subplots(figsize=(max(8, len(layers_x) * 0.5), 5))

ax.plot(layers_x, norm_of_mean,  marker="o", linewidth=2, color="steelblue",
        label=r"$\|\mathbb{E}[\mathbf{v}_{cls}]\|_2$ — norm of mean CLS")
ax.plot(layers_x, mean_of_norms, marker="s", linewidth=2, color="darkorange",
        label=r"$\mathbb{E}[\|\mathbf{v}_{cls}\|_2]$ — mean of CLS norms")

ax.set_xlabel("Layer index", fontsize=11)
ax.set_ylabel("L2 norm", fontsize=11)
ax.set_title(
    f"CLS embedding norms across layers\n{MODEL_NAME}",
    fontsize=11
)
ax.set_xticks(layers_x)
ax.tick_params(axis="x", labelsize=8)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("cls_norm_across_layers.png", dpi=150)
plt.show()
print("Plot saved to cls_norm_across_layers.png")

In [ ]:
# ── Print table: ‖E[cls]‖₂  and  E[‖cls‖₂]  per layer ───────────────────────
print("CLS embedding norms per layer")
print(f"  {'‖E[cls]‖₂':>14} = norm of the mean CLS vector (how strong the consensus direction is)")
print(f"  {'E[‖cls‖₂]':>14} = mean of per-image CLS norms (typical CLS magnitude)")
print(f"  {'ratio':>14} = ‖E[cls]‖₂ / E[‖cls‖₂]  (→ 1 means embeddings are well-aligned)\n")
print(f"{'Layer':>6}  {'‖E[cls]‖₂':>14}  {'E[‖cls‖₂]':>14}  {'ratio':>8}")
print("-" * 52)
for l in TARGET_LAYERS:
    nom   = norm_mean_cls[l]
    mofn  = mean_cls_norm[l]
    ratio = nom / mofn if mofn > 0 else float("nan")
    print(f"{l:>6}  {nom:>14.4f}  {mofn:>14.4f}  {ratio:>8.4f}")

In [ ]:
# ── Print diffs ‖mean_cls[l] − mean_patch[l][i,j]‖₂ per layer ────────────────
# For each layer: print a H×W grid of values, then summary stats.
np.set_printoptions(precision=4, suppress=True, linewidth=120)

print("‖mean_cls[l] − mean_patch[l][i,j]‖₂  (rows = patch rows, cols = patch cols)\n")
for l, hmap in zip(TARGET_LAYERS, all_heatmaps):
    print(f"─── Layer {l} ───")
    print(f"  min={hmap.min():.4f}  max={hmap.max():.4f}  mean={hmap.mean():.4f}  std={hmap.std():.4f}")
    # Print row by row with patch-row labels
    header = "       " + "  ".join(f"{j:6d}" for j in range(W))
    print(header)
    for i in range(H):
        row_str = "  ".join(f"{hmap[i, j]:6.4f}" for j in range(W))
        print(f"  r{i:02d}  {row_str}")
    print()

## Loading `token_mean_embeddings.pt` in future sessions

In [ ]:
import torch

data = torch.load("token_mean_embeddings.pt", map_location="cpu", weights_only=False)

# Keys available:
mean_cls      = data["mean_cls"]       # Tensor [L, d_model]        — mean CLS per layer
mean_patch    = data["mean_patch"]     # Tensor [L, H, W, d_model]  — mean patch per position per layer
norm_mean_cls = data["norm_mean_cls"]  # list[float] len L          — ‖E[cls]‖₂ per layer
mean_cls_norm = data["mean_cls_norm"]  # list[float] len L          — E[‖cls‖₂] per layer
target_layers = data["target_layers"]  # list[int]                  — which layer indices
H, W          = data["patch_grid"]     # e.g. (16, 16)

# Example: get mean CLS at layer 12 (assuming 12 is in target_layers)
layer_pos   = target_layers.index(12)
cls_layer12 = mean_cls[layer_pos]      # Tensor [d_model]

# Example: get mean patch grid at layer 12
patch_layer12 = mean_patch[layer_pos]  # Tensor [H, W, d_model]

# Example: recompute the diff heatmap for layer 12
diff = cls_layer12.unsqueeze(0).unsqueeze(0) - patch_layer12  # [H, W, d_model]
hmap = diff.norm(dim=-1)  # [H, W]